In [1]:
import os
import re

import numpy as np
import torch
import pandas as pd
import random
import matplotlib
import seaborn as sns
import pickle
from tqdm.notebook import tqdm
from scipy.interpolate import interp1d
from functionsgpu_old import *
from plotting_betas import *
import torch
import torch.nn as nn
import torch.nn.functional as F
from val_test import val_test
from video_saving import *

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import xgboost as xgb
from sklearn.preprocessing import MinMaxScaler


import warnings
warnings.filterwarnings("ignore")

device = torch.device('cuda:1' if torch.cuda.is_available() else 'cpu')
print(device)

if device.type == "cuda":
    idx = device.index if device.index is not None else torch.cuda.current_device()
    print(torch.cuda.get_device_name(idx))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Enable deterministic operations (may slow down training slightly)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

tslen = 200

functionsgpu_old.py device: cuda:1
cuda:1
NVIDIA RTX A5000


## Data Loading

In [2]:
def loading(filename, tslen):
    with open('{}/betas_aligned{}.pkl'.format(filename, tslen), 'rb') as f:
        betas_aligned = pickle.load(f)
    with open('{}/mu{}.pkl'.format(filename, tslen), 'rb') as f:
        mu = pickle.load(f)
    with open('{}/tangent_vecs{}.pkl'.format(filename, tslen), 'rb') as f:
        tangent_vec_all = pickle.load(f)
    return betas_aligned, mu, tangent_vec_all

betas_aligned_all, mu_all_t, tangent_vec_all = loading('aligned_data',tslen)
mu_all_t_tensor = torch.from_numpy(mu_all_t).to(device=device, dtype=torch.float32)

print(betas_aligned_all[0].shape, tangent_vec_all.shape, mu_all_t.shape)

(32, 3, 200) (32, 3, 200, 155) (32, 3, 200)


In [3]:
# Create y_lesion from LesionLeft, aligned to same participant_ids order as X

folder_path = "/mnt/sdb/arafat/stroke_riemann/csv_r"
files = [file for file in os.listdir(folder_path)]
files = sorted(files, key=lambda x: int(x.split('_')[0][2:]))    

# Participant ID for each row of X (same order as files from csv_r)
participant_ids = [re.search(r'ID(\d+)_', f).group(1) for f in files]

demo_df = pd.read_csv('demo_data.csv')
id_to_lesion = dict(zip(demo_df['s'].astype(int), demo_df['LesionLeft']))
y_lesion = np.array([id_to_lesion[int(pid)] for pid in participant_ids])

print("LesionLeft class distribution:", np.unique(y_lesion, return_counts=True))
print("y_lesion.shape:", y_lesion.shape)

LesionLeft class distribution: (array([0, 1, 2]), array([ 30,  14, 111]))
y_lesion.shape: (155,)


In [4]:
K = 32
M = 3
T = tslen
nsamples = 155

tangent_flat = tangent_vec_all.reshape((K*M*T, nsamples))
print(tangent_flat.shape)

(19200, 155)


# Nonlinear Tangent VAE

In [5]:
class NonlinearVAE(nn.Module):
    """VAE with the same nonlinear/tied architecture as NonlinearAE.

    Encoder: x -> W1 -> tanh -> dropout -> {W2_mu, W2_logvar} -> {mu, logvar}
    Decoder (tied): z -> W2_mu^T -> tanh -> W1^T -> x_hat
    """
    def __init__(self, D, R, H=32, dropout=0.1):
        super().__init__()
        self.H = H
        # Encoder layers
        self.W1 = nn.Linear(D, H, bias=False)        # input -> hidden
        self.W2_mu = nn.Linear(H, R, bias=False)     # hidden -> latent mean
        self.W2_logvar = nn.Linear(H, R)             # hidden -> latent logvar
        self.dropout = nn.Dropout(p=dropout)

    def encode(self, x):
        h = torch.tanh(self.W1(x))
        h = self.dropout(h)
        mu = self.W2_mu(h)
        logvar = self.W2_logvar(h)
        return mu, logvar

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        # Tied decoder using W2_mu and W1, mirroring NonlinearAE
        h_recon = torch.tanh(F.linear(z, self.W2_mu.weight.T))
        x_hat = F.linear(h_recon, self.W1.weight.T)
        return x_hat

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decode(z)
        return x_hat, mu, logvar, z


def vae_loss(x, x_hat, mu, logvar, beta=1e-4):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon + beta * kl, recon, kl


## Training Function for Each Fold Training (VAE)

In [6]:
def focal_loss(logits, targets, gamma=2.0):
    """Focal loss: - (1 - p_t)^gamma * log(p_t). Down-weights easy examples."""
    log_softmax = F.log_softmax(logits, dim=1)
    ce = F.nll_loss(log_softmax, targets, reduction="none")
    pt = (-ce).exp()
    focal_weight = (1 - pt) ** gamma
    return (focal_weight * ce).mean()


def train_vae_fold_with_aux_clf_early_stopping(
    X_tan_train, y_train, X_tan_val, y_val, D, R, n_classes=3, lambda_clf=0.65,
    num_epochs=2500, lr=1e-3, verbose=False, seed=42, focal_gamma=2,
    patience=150, min_delta=0.002, dtype=torch.float32,
):
    """Train VAE with auxiliary classification head and early stopping.
    Fair comparison with RVAE+EarlyStop.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    aux_head = nn.Linear(R, n_classes).to(device=device, dtype=dtype)
    opt = torch.optim.Adam(list(vae_fold.parameters()) + list(aux_head.parameters()), lr=lr)
    y_t = torch.from_numpy(np.asarray(y_train, dtype=np.int64)).to(device=device)
    y_val_t = torch.from_numpy(np.asarray(y_val, dtype=np.int64)).to(device=device)

    vae_fold.train()
    aux_head.train()
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        opt.zero_grad(set_to_none=True)
        x_hat, mu, logvar, z = vae_fold(X_tan_train)
        loss_vae, recon, kl = vae_loss(X_tan_train, x_hat, mu, logvar, beta=1e-5)
        logits = aux_head(mu)
        loss_clf = focal_loss(logits, y_t, gamma=focal_gamma)
        loss = loss_vae + lambda_clf * loss_clf
        loss.backward()
        opt.step()
        
        # Check validation performance every 50 epochs
        if epoch % 50 == 0:
            vae_fold.eval()
            aux_head.eval()
            with torch.no_grad():
                mu_val, _ = vae_fold.encode(X_tan_val)
                logits_val = aux_head(mu_val)
                val_preds = logits_val.argmax(dim=1).cpu().numpy()
                val_acc = (val_preds == y_val_t.cpu().numpy()).mean()
                
                if val_acc > best_val_acc + min_delta:
                    best_val_acc = val_acc
                    patience_counter = 0
                    best_model_state = {
                        'vae': vae_fold.state_dict(),
                        'aux_head': aux_head.state_dict(),
                    }
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch}, best val acc: {best_val_acc:.4f}")
                    break
            
            vae_fold.train()
            aux_head.train()
    
    # Load best model
    if best_model_state:
        vae_fold.load_state_dict(best_model_state['vae'])
        aux_head.load_state_dict(best_model_state['aux_head'])
    
    vae_fold.eval()
    return vae_fold

## VAE Cross Validation with Early Stopping

In [ ]:
# VAE with Early Stopping
dtype = torch.float32
n_cls = len(y_lesion)
D = tangent_flat.shape[0]
participant_ids_arr = np.asarray(participant_ids)

models_clf = {'KNN': KNeighborsClassifier()}

all_val_clf_vae_early = {name: {'targets': [], 'preds': []} for name in models_clf.keys()}
all_test_clf_vae_early = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models_clf.keys()}

n_folds = 30
R_vae = 38
lambda_clf = 0.65

print("="*80)
print("VAE with EARLY STOPPING")
print(f"Hyperparameters: λ_clf={lambda_clf}, R={R_vae}, Early stopping patience=150")
print("="*80)

for k in tqdm(range(n_folds), total=n_folds, desc='VAE+EarlyStop folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n_cls) if participant_ids_arr[j] in train_pids])
    validation_idx = np.array([j for j in range(n_cls) if participant_ids_arr[j] in validation_pids])
    test_idx = np.array([j for j in range(n_cls) if participant_ids_arr[j] in test_pids])
    
    if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
        continue

    fold_seed = SEED + k
    X_tan_train = torch.from_numpy(tangent_flat[:, train_idx].T.astype(np.float32)).to(device=device, dtype=dtype)
    X_tan_val = torch.from_numpy(tangent_flat[:, validation_idx].T.astype(np.float32)).to(device=device, dtype=dtype)
    y_train_fold_ = y_lesion[train_idx]
    y_val_fold_ = y_lesion[validation_idx]
    
    model_fold = train_vae_fold_with_aux_clf_early_stopping(
        X_tan_train, y_train_fold_, X_tan_val, y_val_fold_, D, R_vae, n_classes=3, 
        lambda_clf=lambda_clf, num_epochs=2500, lr=1e-3, verbose=False, seed=fold_seed,
        patience=150, min_delta=0.002,
    )

    with torch.no_grad():
        mu_train_fold, _ = model_fold.encode(X_tan_train)
        mu_val_fold, _ = model_fold.encode(X_tan_val)
        mu_test_fold, _ = model_fold.encode(torch.from_numpy(tangent_flat[:, test_idx].T.astype(np.float32)).to(device=device, dtype=dtype))

    Z_train_fold = mu_train_fold.cpu().numpy()
    Z_val_fold = mu_val_fold.cpu().numpy()
    Z_test_fold = mu_test_fold.cpu().numpy()
    
    scaler = MinMaxScaler()
    Z_train_fold = scaler.fit_transform(Z_train_fold)
    Z_val_fold = scaler.transform(Z_val_fold)
    Z_test_fold = scaler.transform(Z_test_fold)
    
    y_train_fold = y_lesion[train_idx]
    y_val_fold = y_lesion[validation_idx]
    y_test_fold = y_lesion[test_idx]

    for name, model_clf in models_clf.items():
        m = type(model_clf)(**model_clf.get_params())
        m.fit(Z_train_fold, y_train_fold)
        validation_preds = m.predict(Z_val_fold)
        test_preds = m.predict(Z_test_fold)
        
        all_val_clf_vae_early[name]['targets'].extend(y_val_fold.tolist())
        all_val_clf_vae_early[name]['preds'].extend(validation_preds.tolist())
        all_test_clf_vae_early[name]['targets'].extend(y_test_fold.tolist())
        all_test_clf_vae_early[name]['preds'].extend(test_preds.tolist())
        all_test_clf_vae_early[name]['subjects'].extend(participant_ids_arr[test_idx].tolist())
        
        acc_val = accuracy_score(y_val_fold, validation_preds)
        f1_val = f1_score(y_val_fold, validation_preds, average='weighted')
        print(f"Fold {k + 1:02d} | {name} | Validation: Accuracy={acc_val:.3f}, F1={f1_val:.3f}")

print("\n=== VAE+EarlyStop — Validation Performance ===")
results_val_clf_vae_early = {}
for name in models_clf.keys():
    t = np.array(all_val_clf_vae_early[name]['targets'])
    p = np.array(all_val_clf_vae_early[name]['preds'])
    if t.size == 0:
        continue
    results_val_clf_vae_early[name] = {
        'Accuracy': accuracy_score(t, p),
        'F1 (weighted)': f1_score(t, p, average='weighted'),
        'F1 (macro)': f1_score(t, p, average='macro'),
    }
print(pd.DataFrame(results_val_clf_vae_early).T)

print("\n=== VAE+EarlyStop — Test Performance ===")
results_clf_vae_early = {}
for name in models_clf.keys():
    all_targets = np.array(all_test_clf_vae_early[name]['targets'])
    all_preds = np.array(all_test_clf_vae_early[name]['preds'])
    if all_targets.size == 0:
        continue
    results_clf_vae_early[name] = {
        'Accuracy': accuracy_score(all_targets, all_preds),
        'F1 (weighted)': f1_score(all_targets, all_preds, average='weighted'),
        'F1 (macro)': f1_score(all_targets, all_preds, average='macro'),
        'Precision (weighted)': precision_score(all_targets, all_preds, average='weighted', zero_division=0),
        'Precision (macro)': precision_score(all_targets, all_preds, average='macro'),
        'Recall (weighted)': recall_score(all_targets, all_preds, average='weighted', zero_division=0),
        'Recall (macro)': recall_score(all_targets, all_preds, average='macro'),
    }
results_clf_vae_early_df = pd.DataFrame(results_clf_vae_early).T
results_clf_vae_early_df

In [ ]:
from ci_class import subject_bootstrap_ci_class

ci_results = {}

name = "KNN"

ci_results[name] = subject_bootstrap_ci_class(
    all_test_clf_vae_early[name]['targets'],
    all_test_clf_vae_early[name]['preds'],
    all_test_clf_vae_early[name]['subjects'])

pd.DataFrame(ci_results['KNN'])

,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
mean,0.896774,0.897159,0.809759,0.900611,0.829234,0.896774,0.799271
ci,"[0.862, 0.936]","[0.862, 0.935]","[0.723, 0.881]","[0.866, 0.94]","[0.748, 0.907]","[0.862, 0.936]","[0.716, 0.882]"


# KT-RSV

In [ ]:
class KendallRVAE(nn.Module):
    """rVAE: encode tangent vectors, decode to tangent, but (during
    training) compare reconstructions on the manifold via an exp map.

    - Inputs: tangent vectors (N, D) as in the VAE section.
    - Decoder: produces tangent vectors.
    - Training loss: uses expmap(mu, v_hat) vs. original manifold trajectory.
    """
    def __init__(self, base_vae, mu_shape, expmap):
        super().__init__()
        self.vae = base_vae
        # mean shape, used for exponential map when mapping back to manifold
        self.register_buffer("mu_shape", mu_shape)
        self.expmap = expmap

    def forward(self, x):
        """Forward on tangent vectors.

        x : (N, D) tangent vectors
        Returns
        -------
        x_man_hat : (N, D) manifold trajectory flattened (via expmap)
        mu_z, logvar, z, v_hat : usual VAE outputs (in tangent space)
        """
        v_hat, mu_z, logvar, z = self.vae(x)   # tangent reconstruction

        # Map reconstructed tangent field back to the manifold
        B = v_hat.shape[0]
        v_hat_reshaped = v_hat.view(B, K, M, T)
        mu = self.mu_shape.view(K, M, T)
        x_recon_man = self.expmap(mu, v_hat_reshaped)   # (B, K, M, T)
        x_recon_man = x_recon_man.view(B, -1)

        return x_recon_man, mu_z, logvar, z, v_hat

## Getting Orginal Trajectories on Manifold

In [ ]:
betas = np.array(betas_aligned_all)
print(betas.shape)   # (155, 32, 3, 200)

N, K, M, T = betas.shape   # correct order

# Shape mean as before (kept for potential manifold utilities)
mu_shape = torch.from_numpy(
    mu_all_t.reshape(-1).astype(np.float32)
).to(device)

print("mu_shape:", mu_shape.shape)      # should be (19200,)

(155, 32, 3, 200)
mu_shape: torch.Size([19200])


## KT-RSV Training on Full Dataset (test run)

In [ ]:
dtype = torch.float32

# Tangent input (as in the VAE section)
X_tan = torch.from_numpy(tangent_flat.T.astype(np.float32)).to(device=device, dtype=dtype)
mu_shape = mu_shape.to(device=device, dtype=dtype)

# Corresponding manifold trajectories (flattened)
betas = np.array(betas_aligned_all)              # (N, K, M, T)
X_man = torch.from_numpy(betas.reshape(betas.shape[0], -1).astype(np.float32)).to(device=device, dtype=dtype)

D = X_tan.shape[1]
R = 10

base_vae = NonlinearVAE(D, R).to(device=device, dtype=dtype)
model = KendallRVAE(base_vae, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(3000):
    opt.zero_grad(set_to_none=True)

    # forward in tangent space, loss in manifold space
    x_hat_man, mu, logvar, z, v_hat = model(X_tan)

    # Geodesic reconstruction loss on the manifold
    dist = squared_geodesic_distance(X_man, x_hat_man, K, M, T)
    recon = torch.mean(dist)

    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    loss = recon + 1e-6 * kl

    loss.backward()
    opt.step()

    if epoch % 200 == 0:
        print(f"Epoch {epoch} | loss {loss:.6f} | recon {recon:.6f} | kl {kl:.6f}")

Epoch 0 | loss 0.018341 | recon 0.018341 | kl 0.002982
Epoch 200 | loss 0.012238 | recon 0.012238 | kl 0.059873
Epoch 400 | loss 0.009736 | recon 0.009735 | kl 0.238723
Epoch 600 | loss 0.009165 | recon 0.009165 | kl 0.396473
Epoch 800 | loss 0.008554 | recon 0.008554 | kl 0.574966
Epoch 1000 | loss 0.008258 | recon 0.008257 | kl 0.730885
Epoch 1200 | loss 0.008184 | recon 0.008183 | kl 0.898655
Epoch 1400 | loss 0.006880 | recon 0.006878 | kl 1.556337
Epoch 1600 | loss 0.005928 | recon 0.005926 | kl 2.388893
Epoch 1800 | loss 0.004899 | recon 0.004896 | kl 3.021168
Epoch 2000 | loss 0.004765 | recon 0.004761 | kl 3.341730
Epoch 2200 | loss 0.004744 | recon 0.004741 | kl 3.490101
Epoch 2400 | loss 0.004876 | recon 0.004872 | kl 3.536266
Epoch 2600 | loss 0.004694 | recon 0.004690 | kl 3.661430
Epoch 2800 | loss 0.004686 | recon 0.004682 | kl 3.784869


## Training Fuction for KT-RSV with Early Stopping

In [ ]:
def train_rvae_with_classification_early_stopping(
    X_tan_train, X_man_train, y_train, X_tan_val, y_val, K, M, T,
    R=38, lambda_clf=1.2, alpha_recon=0.5, num_epochs=2500,
    patience=150, min_delta=0.002, verbose=False, seed=42
):
    """Train RVAE with early stopping based on validation classification accuracy.
    This is FAIR because it may use FEWER epochs than VAE (not more).
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    D = X_tan_train.shape[1]
    base_vae_fold = NonlinearVAE(D, R).to(device=device, dtype=dtype)
    model_fold = KendallRVAE(base_vae_fold, mu_shape, exp_gpu_batch).to(device=device, dtype=dtype)
    aux_head = nn.Linear(R, 3).to(device=device, dtype=dtype)
    opt_fold = torch.optim.Adam(list(model_fold.parameters()) + list(aux_head.parameters()), lr=1e-3)
    y_t = torch.from_numpy(np.asarray(y_train, dtype=np.int64)).to(device=device)
    y_val_t = torch.from_numpy(np.asarray(y_val, dtype=np.int64)).to(device=device)

    model_fold.train()
    aux_head.train()
    
    best_val_acc = 0.0
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        opt_fold.zero_grad(set_to_none=True)
        x_hat_man_train, mu_train, logvar_train, z_train, v_hat_train = model_fold(X_tan_train)
        dist_train = squared_geodesic_distance(X_man_train, x_hat_man_train, K, M, T)
        recon_train = torch.mean(dist_train)
        kl_train = -0.5 * torch.mean(1 + logvar_train - mu_train.pow(2) - logvar_train.exp())
        loss_rvae = alpha_recon * recon_train + 1e-6 * kl_train
        logits = aux_head(mu_train)
        loss_clf = focal_loss(logits, y_t, gamma=2.0)
        loss_train = loss_rvae + lambda_clf * loss_clf
        loss_train.backward()
        opt_fold.step()
        
        # Check validation performance every 50 epochs
        if epoch % 50 == 0:
            model_fold.eval()
            aux_head.eval()
            with torch.no_grad():
                _, mu_val, _, _, _ = model_fold(X_tan_val)
                logits_val = aux_head(mu_val)
                val_preds = logits_val.argmax(dim=1).cpu().numpy()
                val_acc = (val_preds == y_val_t.cpu().numpy()).mean()
                
                if val_acc > best_val_acc + min_delta:
                    best_val_acc = val_acc
                    patience_counter = 0
                    best_model_state = {
                        'model': model_fold.state_dict(),
                        'aux_head': aux_head.state_dict(),
                    }
                else:
                    patience_counter += 1
                    
                if patience_counter >= patience:
                    if verbose:
                        print(f"Early stopping at epoch {epoch}, best val acc: {best_val_acc:.4f}")
                    break
            
            model_fold.train()
            aux_head.train()
    
    # Load best model
    if best_model_state:
        model_fold.load_state_dict(best_model_state['model'])
        aux_head.load_state_dict(best_model_state['aux_head'])
    
    model_fold.eval()
    return model_fold

## KT-RSV Training Loop

In [ ]:
# RVAE with EARLY STOPPING - Alternative Fair Improvement
# Uses early stopping (may use fewer epochs)

dtype = torch.float32
n = len(y_lesion)
participant_ids_arr = np.asarray(participant_ids)

models_clf_rvae = models_clf
all_val_clf_rvae_early = {name: {'targets': [], 'preds': []} for name in models_clf_rvae.keys()}
all_test_clf_rvae_early = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models_clf_rvae.keys()}

n_folds = 30

R_rvae = R_vae  # 38 (same as VAE)
lambda_clf_rvae = 1.2
alpha_recon_rvae = 0.5

print("="*80)
print("KT-RSV with EARLY STOPPING")
print(f"Hyperparameters: λ_clf={lambda_clf_rvae}, α_recon={alpha_recon_rvae}, R={R_rvae}")
print("Early stopping: stops if validation accuracy doesn't improve for 150 epochs (50 epoch checks)")
print("="*80)

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV+EarlyStop folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids_arr[j] in train_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids_arr[j] in validation_pids])
    test_idx = np.array([j for j in range(n) if participant_ids_arr[j] in test_pids])

    if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
        continue

    fold_seed = SEED + k
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)

    X_tan_train = X_tan[train_idx]
    X_man_train = X_man[train_idx]
    X_tan_val = X_tan[validation_idx]
    y_train_fold_ = y_lesion[train_idx]
    y_val_fold_ = y_lesion[validation_idx]

    # Train RVAE with early stopping
    model_fold = train_rvae_with_classification_early_stopping(
        X_tan_train, X_man_train, y_train_fold_, X_tan_val, y_val_fold_, K, M, T,
        R=R_rvae, lambda_clf=lambda_clf_rvae, alpha_recon=alpha_recon_rvae,
        num_epochs=2500, patience=150, min_delta=0.002, verbose=False, seed=fold_seed
    )

    # Extract latent means
    with torch.no_grad():
        mu_train_fold, _ = model_fold.vae.encode(X_tan_train)
        mu_val_fold, _ = model_fold.vae.encode(X_tan_val)
        mu_test_fold, _ = model_fold.vae.encode(X_tan[test_idx])

    Z_train_fold = mu_train_fold.detach().cpu().numpy()
    Z_val_fold = mu_val_fold.detach().cpu().numpy()
    Z_test_fold = mu_test_fold.detach().cpu().numpy()

    # Scale latent per fold
    scaler = MinMaxScaler()
    Z_train_fold = scaler.fit_transform(Z_train_fold)
    Z_val_fold = scaler.transform(Z_val_fold)
    Z_test_fold = scaler.transform(Z_test_fold)

    y_train_fold = y_lesion[train_idx]
    y_val_fold = y_lesion[validation_idx]
    y_test_fold = y_lesion[test_idx]

    # Train and evaluate each classifier
    for name, model_clf in models_clf_rvae.items():
        m = type(model_clf)(**model_clf.get_params())
        m.fit(Z_train_fold, y_train_fold)
        val_preds = m.predict(Z_val_fold)
        test_preds = m.predict(Z_test_fold)

        all_val_clf_rvae_early[name]['targets'].extend(y_val_fold.tolist())
        all_val_clf_rvae_early[name]['preds'].extend(val_preds.tolist())
        all_test_clf_rvae_early[name]['targets'].extend(y_test_fold.tolist())
        all_test_clf_rvae_early[name]['preds'].extend(test_preds.tolist())
        all_test_clf_rvae_early[name]['subjects'].extend(participant_ids_arr[test_idx].tolist())

        acc_val = accuracy_score(y_val_fold, val_preds)
        f1_val = f1_score(y_val_fold, val_preds, average='weighted')
        print(f"Fold {k + 1:02d} | {name} | Validation: Accuracy={acc_val:.3f}, F1={f1_val:.3f}")

print("\n=== RVAE+EarlyStop — Validation Performance ===")
results_val_clf_rvae_early = {}
for name in models_clf_rvae.keys():
    t = np.array(all_val_clf_rvae_early[name]['targets'])
    p = np.array(all_val_clf_rvae_early[name]['preds'])
    if t.size == 0:
        continue
    results_val_clf_rvae_early[name] = {
        'Accuracy': accuracy_score(t, p),
        'F1 (weighted)': f1_score(t, p, average='weighted'),
        'F1 (macro)': f1_score(t, p, average='macro'),
    }
print(pd.DataFrame(results_val_clf_rvae_early).T)

print("\n=== RVAE+EarlyStop — Test Performance ===")
results_clf_rvae_early = {}
predictions_clf = {}
for name in models_clf_rvae.keys():
    all_targets = np.array(all_test_clf_rvae_early[name]['targets'])
    all_preds = np.array(all_test_clf_rvae_early[name]['preds'])
    if all_targets.size == 0:
        continue
    predictions_clf[name] = {'y_true': all_targets, 'y_pred': all_preds}
    results_clf_rvae_early[name] = {
        'Accuracy': accuracy_score(all_targets, all_preds),
        'F1 (weighted)': f1_score(all_targets, all_preds, average='weighted'),
        'F1 (macro)': f1_score(all_targets, all_preds, average='macro'),
    }
results_clf_rvae_early_df = pd.DataFrame(results_clf_rvae_early).T
results_clf_rvae_early_df

RVAE with EARLY STOPPING
Hyperparameters: λ_clf=1.2, α_recon=0.5, R=38
Early stopping: stops if validation accuracy doesn't improve for 150 epochs (50 epoch checks)


RVAE+EarlyStop folds:   0%|          | 0/30 [00:00<?, ?it/s]

Fold 01 | KNN | Validation: Accuracy=0.800, F1=0.886
Fold 01 | SVM | Validation: Accuracy=0.800, F1=0.886
Fold 01 | RF | Validation: Accuracy=0.800, F1=0.886
Fold 01 | XGBoost | Validation: Accuracy=0.800, F1=0.886
Fold 01 | MLP | Validation: Accuracy=0.800, F1=0.886
Fold 02 | KNN | Validation: Accuracy=0.600, F1=0.667
Fold 02 | SVM | Validation: Accuracy=0.600, F1=0.667
Fold 02 | RF | Validation: Accuracy=0.600, F1=0.667
Fold 02 | XGBoost | Validation: Accuracy=0.400, F1=0.343
Fold 02 | MLP | Validation: Accuracy=0.600, F1=0.667
Fold 03 | KNN | Validation: Accuracy=0.800, F1=0.800
Fold 03 | SVM | Validation: Accuracy=0.800, F1=0.800
Fold 03 | RF | Validation: Accuracy=0.800, F1=0.800
Fold 03 | XGBoost | Validation: Accuracy=0.800, F1=0.800
Fold 03 | MLP | Validation: Accuracy=0.800, F1=0.800
Fold 04 | KNN | Validation: Accuracy=0.600, F1=0.567
Fold 04 | SVM | Validation: Accuracy=0.600, F1=0.567
Fold 04 | RF | Validation: Accuracy=0.600, F1=0.567
Fold 04 | XGBoost | Validation: Accura

,Accuracy,F1 (weighted),F1 (macro)
KNN,0.909677,0.909274,0.831738
SVM,0.890323,0.888883,0.780636
RF,0.877419,0.873496,0.730294
XGBoost,0.877419,0.861925,0.714089
MLP,0.890323,0.881939,0.735783


In [ ]:
from ci_class import subject_bootstrap_ci_class

ci_results_rvae = {}

name = "KNN"

ci_results_rvae[name] = subject_bootstrap_ci_class(
    all_test_clf_rvae_early[name]['targets'],
    all_test_clf_rvae_early[name]['preds'],
    all_test_clf_rvae_early[name]['subjects'])

pd.DataFrame(ci_results_rvae['KNN'])

,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
mean,0.909677,0.909274,0.831738,0.913928,0.866419,0.909677,0.813385
ci,"[0.876, 0.947]","[0.875, 0.946]","[0.754, 0.903]","[0.881, 0.95]","[0.804, 0.93]","[0.876, 0.947]","[0.735, 0.895]"


In [ ]:
from sklearn.metrics import classification_report

# Classification report for each model
target_names = ['LesionRight (0)', 'LesionLeft (1)', 'Healthy (2)']
for name in predictions_clf:
    d = predictions_clf[name]
    print(f"\n=== {name} ===")
    print(classification_report(d['y_true'], d['y_pred'], target_names=target_names, zero_division=0))


=== KNN ===
                 precision    recall  f1-score   support

LesionRight (0)       0.74      0.83      0.78        30
 LesionLeft (1)       0.90      0.64      0.75        14
    Healthy (2)       0.96      0.96      0.96       111

       accuracy                           0.91       155
      macro avg       0.87      0.81      0.83       155
   weighted avg       0.91      0.91      0.91       155


=== SVM ===
                 precision    recall  f1-score   support

LesionRight (0)       0.68      0.83      0.75        30
 LesionLeft (1)       0.88      0.50      0.64        14
    Healthy (2)       0.96      0.95      0.96       111

       accuracy                           0.89       155
      macro avg       0.84      0.76      0.78       155
   weighted avg       0.90      0.89      0.89       155


=== RF ===
                 precision    recall  f1-score   support

LesionRight (0)       0.66      0.90      0.76        30
 LesionLeft (1)       0.71      0.36      0

In [ ]:
z = pd.read_csv('/mnt/sdb/arafat/stroke_riemann/result_figures/1e-6/zdf_R38.csv') # 2^-1, R=5
z = z.drop(columns=['c']).values
print(z.shape, len(participant_ids))    

(155, 38) 155


In [ ]:
# RVAE with EARLY STOPPING - Alternative Fair Improvement
# Uses early stopping (may use fewer epochs)
import random
import numpy as np
import torch

from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
import xgboost as xgb
from val_test import val_test
from print_results import print_results
from tqdm.notebook import tqdm


# Regression models with random_state set for reproducibility
models_clf = {
    'KNN': KNeighborsClassifier(),
    'SVM': SVC(kernel='rbf'),
    'RF': RandomForestClassifier(random_state=42),
    'XGBoost': xgb.XGBClassifier(random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42),
}


dtype = torch.float32
n = len(y_lesion)
participant_ids_arr = np.asarray(participant_ids)

models_clf_rvae = models_clf
all_val_clf_rvae_early = {name: {'targets': [], 'preds': []} for name in models_clf_rvae.keys()}
all_test_clf_rvae_early = {name: {'targets': [], 'preds': [], 'subjects': []} for name in models_clf_rvae.keys()}

n_folds = 30

for k in tqdm(range(n_folds), total=n_folds, desc='KT-RSV folds'):
    validation_pids_list, test_pids_list = val_test(participant_ids, k)
    validation_pids = set(validation_pids_list)
    test_pids = set(test_pids_list)
    train_pids = set(participant_ids) - validation_pids - test_pids

    train_idx = np.array([j for j in range(n) if participant_ids_arr[j] in train_pids])
    validation_idx = np.array([j for j in range(n) if participant_ids_arr[j] in validation_pids])
    test_idx = np.array([j for j in range(n) if participant_ids_arr[j] in test_pids])

    if len(train_idx) == 0 or len(validation_idx) == 0 or len(test_idx) == 0:
        continue

    fold_seed = SEED + k
    random.seed(fold_seed)
    np.random.seed(fold_seed)
    torch.manual_seed(fold_seed)
    torch.cuda.manual_seed_all(fold_seed)

    Z_train_fold = z[train_idx]
    Z_val_fold = z[validation_idx]
    Z_test_fold = z[test_idx]

    # Scale latent per fold
    # scaler = MinMaxScaler()
    # Z_train_fold = scaler.fit_transform(Z_train_fold)
    # Z_val_fold = scaler.transform(Z_val_fold)
    # Z_test_fold = scaler.transform(Z_test_fold)

    y_train_fold = y_lesion[train_idx]
    y_val_fold = y_lesion[validation_idx]
    y_test_fold = y_lesion[test_idx]

    # Train and evaluate each classifier
    for name, model_clf in models_clf_rvae.items():
        m = type(model_clf)(**model_clf.get_params())
        m.fit(Z_train_fold, y_train_fold)
        val_preds = m.predict(Z_val_fold)
        test_preds = m.predict(Z_test_fold)

        all_val_clf_rvae_early[name]['targets'].extend(y_val_fold.tolist())
        all_val_clf_rvae_early[name]['preds'].extend(val_preds.tolist())
        all_test_clf_rvae_early[name]['targets'].extend(y_test_fold.tolist())
        all_test_clf_rvae_early[name]['preds'].extend(test_preds.tolist())
        all_test_clf_rvae_early[name]['subjects'].extend(participant_ids_arr[test_idx].tolist())

        acc_val = accuracy_score(y_val_fold, val_preds)
        f1_val = f1_score(y_val_fold, val_preds, average='weighted')
        # print(f"Fold {k + 1:02d} | {name} | Validation: Accuracy={acc_val:.3f}, F1={f1_val:.3f}")

print("\n=== KT-RSV Validation Performance ===")
results_val_clf_rvae_early = {}
for name in models_clf_rvae.keys():
    t = np.array(all_val_clf_rvae_early[name]['targets'])
    p = np.array(all_val_clf_rvae_early[name]['preds'])
    if t.size == 0:
        continue
    results_val_clf_rvae_early[name] = {
        'Accuracy': accuracy_score(t, p),
        'F1 (weighted)': f1_score(t, p, average='weighted'),
        'F1 (macro)': f1_score(t, p, average='macro'),
    }
print(pd.DataFrame(results_val_clf_rvae_early).T)

print("\n=== KT-RSV Test Performance ===")
results_clf_rvae_early = {}
predictions_clf = {}
for name in models_clf_rvae.keys():
    all_targets = np.array(all_test_clf_rvae_early[name]['targets'])
    all_preds = np.array(all_test_clf_rvae_early[name]['preds'])
    if all_targets.size == 0:
        continue
    predictions_clf[name] = {'y_true': all_targets, 'y_pred': all_preds}
    results_clf_rvae_early[name] = {
        'Accuracy': accuracy_score(all_targets, all_preds),
        'F1 (weighted)': f1_score(all_targets, all_preds, average='weighted'),
        'F1 (macro)': f1_score(all_targets, all_preds, average='macro'),
    }
results_clf_rvae_early_df = pd.DataFrame(results_clf_rvae_early).T
results_clf_rvae_early_df

KT-RSV folds:   0%|          | 0/30 [00:00<?, ?it/s]


=== KT-RSV Validation Performance ===
         Accuracy  F1 (weighted)  F1 (macro)
KNN      0.929032       0.925475    0.873326
SVM      0.890323       0.882514    0.766025
RF       0.877419       0.868930    0.709335
XGBoost  0.877419       0.869209    0.690869
MLP      0.916129       0.913438    0.831903

=== KT-RSV Test Performance ===


,Accuracy,F1 (weighted),F1 (macro)
KNN,0.929032,0.925475,0.873326
SVM,0.890323,0.882514,0.766025
RF,0.877419,0.868930,0.709335
XGBoost,0.877419,0.869209,0.690869
MLP,0.916129,0.913438,0.831903


In [ ]:
from ci_class import subject_bootstrap_ci_class

ci_results_rvae = {}

name = "KNN"

ci_results_rvae[name] = subject_bootstrap_ci_class(
    all_test_clf_rvae_early[name]['targets'],
    all_test_clf_rvae_early[name]['preds'],
    all_test_clf_rvae_early[name]['subjects'])

pd.DataFrame(ci_results_rvae['KNN'])

,Accuracy,F1 (weighted),F1 (macro),Precision (weighted),Precision (macro),Recall (weighted),Recall (macro)
mean,0.929032,0.925475,0.873326,0.929524,0.939129,0.929032,0.826984
ci,"[0.898, 0.96]","[0.891, 0.958]","[0.805, 0.935]","[0.899, 0.96]","[0.906, 0.977]","[0.898, 0.96]","[0.75, 0.905]"


In [ ]:
from sklearn.metrics import classification_report

name = "KNN"

# Classification report for each model
target_names = ['LesionRight (0)', 'LesionLeft (1)', 'Healthy (2)']
d = predictions_clf[name]
print(f"\n=== {name} ===")
print(classification_report(d['y_true'], d['y_pred'], target_names=target_names, zero_division=0))


=== KNN ===
                 precision    recall  f1-score   support

LesionRight (0)       0.88      0.77      0.82        30
 LesionLeft (1)       1.00      0.71      0.83        14
    Healthy (2)       0.93      1.00      0.97       111

       accuracy                           0.93       155
      macro avg       0.94      0.83      0.87       155
   weighted avg       0.93      0.93      0.93       155

